## **FİLTRE TABANLI ÖZNİTELİK SEÇİM YÖNTEMLERİ**

In [25]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import (
    SelectKBest,
    f_classif,
    chi2,
    mutual_info_classif
)

from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, mutual_info_score

import warnings
warnings.filterwarnings("ignore")

In [26]:
RANDOM_STATE = 42

K_VALUES = [
    10, 20, 30, 50, 75,
    100, 150, 200, 300,
    400, 500
]

CV = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

print("K değerleri:", K_VALUES)

K değerleri: [10, 20, 30, 50, 75, 100, 150, 200, 300, 400, 500]


In [27]:
train_path = "/kaggle/input/datasets/crawford/gene-expression/data_set_ALL_AML_train.csv"
actual_path = "/kaggle/input/datasets/crawford/gene-expression/actual.csv"

train_df = pd.read_csv(train_path)
actual_df = pd.read_csv(actual_path)

print("Train veri boyutu:", train_df.shape)
print("Actual veri boyutu:", actual_df.shape)

display(train_df.head())
display(actual_df.head())

Train veri boyutu: (7129, 78)
Actual veri boyutu: (72, 2)


,Gene Description,Gene Accession Number,1,call,2,call.1,3,call.2,4,call.3,...,29,call.33,30,call.34,31,call.35,32,call.36,33,call.37
0,AFFX-BioB-5_at (endogenous control),AFFX-BioB-5_at,-214,A,-139,A,-76,A,-135,A,...,15,A,-318,A,-32,A,-124,A,-135,A
1,AFFX-BioB-M_at (endogenous control),AFFX-BioB-M_at,-153,A,-73,A,-49,A,-114,A,...,-114,A,-192,A,-49,A,-79,A,-186,A
2,AFFX-BioB-3_at (endogenous control),AFFX-BioB-3_at,-58,A,-1,A,-307,A,265,A,...,2,A,-95,A,49,A,-37,A,-70,A
3,AFFX-BioC-5_at (endogenous control),AFFX-BioC-5_at,88,A,283,A,309,A,12,A,...,193,A,312,A,230,P,330,A,337,A
4,AFFX-BioC-3_at (endogenous control),AFFX-BioC-3_at,-295,A,-264,A,-376,A,-419,A,...,-51,A,-139,A,-367,A,-188,A,-407,A


,patient,cancer
0,1,ALL
1,2,ALL
2,3,ALL
3,4,ALL
4,5,ALL


In [28]:
patient_columns = [
    col for col in train_df.columns
    if str(col).isdigit()
]

print("Hasta sayısı:", len(patient_columns))
print("Hasta numaraları:", patient_columns)

Hasta sayısı: 38
Hasta numaraları: ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '34', '35', '36', '37', '38', '28', '29', '30', '31', '32', '33']


In [29]:
# Gen ekspresyon değerleri
X_train_raw = train_df[patient_columns].apply(
    pd.to_numeric,
    errors="coerce"
)

# Genler -> satırlar, hastalar -> sütunlar
X_train = X_train_raw.T

# Hasta numaralarını integer yap
X_train.index = X_train.index.astype(int)

print("X boyutu:", X_train.shape)

display(X_train.iloc[:5, :5])

X boyutu: (38, 7129)


,0,1,2,3,4
1,-214,-153,-58,88,-295
2,-139,-73,-1,283,-264
3,-76,-49,-307,309,-376
4,-135,-114,265,12,-419
5,-106,-125,-76,168,-230


In [30]:
actual_df["patient"] = pd.to_numeric(
    actual_df["patient"],
    errors="coerce"
)

# Sadece train dosyasında bulunan hastaları al
y_df = actual_df[
    actual_df["patient"].isin(X_train.index)
].copy()

# Hasta numarasını index yap
y_df = y_df.set_index("patient")

# X ile aynı hasta sırasına getir
y_df = y_df.loc[X_train.index]

y = y_df["cancer"]

print("Hasta sayısı:", len(y))
print("\nSınıf dağılımı:")
print(y.value_counts())

Hasta sayısı: 38

Sınıf dağılımı:
cancer
ALL    27
AML    11
Name: count, dtype: int64


In [31]:
label_mapping = {
    "ALL": 0,
    "AML": 1
}

y_encoded = y.map(label_mapping)

print(y_encoded.value_counts())
print(y_encoded.head())

cancer
0    27
1    11
Name: count, dtype: int64
1    0
2    0
3    0
4    0
5    0
Name: cancer, dtype: int64


In [32]:
def information_gain_score(X, y):
    X = np.asarray(X)
    y = np.asarray(y)

    n_features = X.shape[1]
    scores = np.zeros(n_features)

    for j in range(n_features):

        feature = X[:, j]

        # Eksik değerleri medyan ile doldur
        if np.isnan(feature).any():
            feature = np.nan_to_num(
                feature,
                nan=np.nanmedian(feature)
            )

        # Sabit özellik
        if np.ptp(feature) == 0:
            scores[j] = 0
            continue

        # Quantile binning
        try:
            bins = np.quantile(
                feature,
                [0, 0.2, 0.4, 0.6, 0.8, 1.0]
            )

            bins = np.unique(bins)

            if len(bins) < 3:
                scores[j] = 0
                continue

            feature_discrete = np.digitize(
                feature,
                bins[1:-1],
                right=False
            )

            # Mutual information / Information Gain
            scores[j] = mutual_info_score(
                y,
                feature_discrete
            )

        except Exception:
            scores[j] = 0

    return scores, np.ones(n_features)

In [33]:
def evaluate_feature_selection(
    X,
    y,
    method_name,
    score_function,
    k_values=K_VALUES,
    cv=CV
):

    results = []

    for k in k_values:

        # k veri setindeki özellik sayısından büyükse atla
        if k > X.shape[1]:
            continue

        pipeline = Pipeline([
            (
                "scaler",
                MinMaxScaler()
            ),

            (
                "feature_selection",
                SelectKBest(
                    score_func=score_function,
                    k=k
                )
            ),

            (
                "classifier",
                LinearSVC(
                    C=1.0,
                    random_state=RANDOM_STATE,
                    max_iter=10000
                )
            )
        ])

        scores = cross_val_score(
            pipeline,
            X,
            y,
            cv=cv,
            scoring="accuracy",
            n_jobs=-1
        )

        results.append({
            "Method": method_name,
            "k": k,
            "CV_Mean_Accuracy": scores.mean(),
            "CV_Std": scores.std()
        })

        print(
            f"{method_name:25s} "
            f"k={k:3d} | "
            f"Accuracy={scores.mean():.4f} "
            f"+/- {scores.std():.4f}"
        )

    results_df = pd.DataFrame(results)

    best_row = results_df.loc[
        results_df["CV_Mean_Accuracy"].idxmax()
    ]

    return results_df, best_row

In [34]:
anova_results_train, anova_best_train = evaluate_feature_selection(
    X_train,
    y_encoded,
    method_name="ANOVA",
    score_function=f_classif
)

print("\nANOVA EN İYİ SONUÇ")
print(anova_best_train)

ANOVA                     k= 10 | Accuracy=0.8714 +/- 0.0794
ANOVA                     k= 20 | Accuracy=0.9214 +/- 0.0645
ANOVA                     k= 30 | Accuracy=0.9214 +/- 0.0645
ANOVA                     k= 50 | Accuracy=0.9500 +/- 0.0612
ANOVA                     k= 75 | Accuracy=0.9214 +/- 0.0645
ANOVA                     k=100 | Accuracy=0.9500 +/- 0.0612
ANOVA                     k=150 | Accuracy=0.9750 +/- 0.0500
ANOVA                     k=200 | Accuracy=0.9750 +/- 0.0500
ANOVA                     k=300 | Accuracy=0.9500 +/- 0.0612
ANOVA                     k=400 | Accuracy=0.9500 +/- 0.1000
ANOVA                     k=500 | Accuracy=0.9500 +/- 0.1000

ANOVA EN İYİ SONUÇ
Method              ANOVA
k                     150
CV_Mean_Accuracy    0.975
CV_Std               0.05
Name: 6, dtype: object


In [35]:
display(
    anova_results_train.sort_values(
        "CV_Mean_Accuracy",
        ascending=False
    )
)

,Method,k,CV_Mean_Accuracy,CV_Std
6,ANOVA,150,0.975000,0.050000
7,ANOVA,200,0.975000,0.050000
5,ANOVA,100,0.950000,0.061237
9,ANOVA,400,0.950000,0.100000
8,ANOVA,300,0.950000,0.061237
10,ANOVA,500,0.950000,0.100000
3,ANOVA,50,0.950000,0.061237
1,ANOVA,20,0.921429,0.064484
2,ANOVA,30,0.921429,0.064484
4,ANOVA,75,0.921429,0.064484


In [36]:
chi2_results_train, chi2_best_train = evaluate_feature_selection(
    X_train,
    y_encoded,
    method_name="Chi-Square",
    score_function=chi2
)

print("\nCHI-SQUARE EN İYİ SONUÇ")
print(chi2_best_train)

Chi-Square                k= 10 | Accuracy=0.8679 +/- 0.0795
Chi-Square                k= 20 | Accuracy=0.8964 +/- 0.0948
Chi-Square                k= 30 | Accuracy=0.9214 +/- 0.0645
Chi-Square                k= 50 | Accuracy=0.9714 +/- 0.0571
Chi-Square                k= 75 | Accuracy=0.9750 +/- 0.0500
Chi-Square                k=100 | Accuracy=0.9500 +/- 0.0612
Chi-Square                k=150 | Accuracy=0.9500 +/- 0.0612
Chi-Square                k=200 | Accuracy=0.9750 +/- 0.0500
Chi-Square                k=300 | Accuracy=1.0000 +/- 0.0000
Chi-Square                k=400 | Accuracy=1.0000 +/- 0.0000
Chi-Square                k=500 | Accuracy=0.9750 +/- 0.0500

CHI-SQUARE EN İYİ SONUÇ
Method              Chi-Square
k                          300
CV_Mean_Accuracy           1.0
CV_Std                     0.0
Name: 8, dtype: object


In [37]:
display(
    chi2_results_train.sort_values(
        "CV_Mean_Accuracy",
        ascending=False
    )
)

,Method,k,CV_Mean_Accuracy,CV_Std
9,Chi-Square,400,1.000000,0.000000
8,Chi-Square,300,1.000000,0.000000
4,Chi-Square,75,0.975000,0.050000
10,Chi-Square,500,0.975000,0.050000
7,Chi-Square,200,0.975000,0.050000
3,Chi-Square,50,0.971429,0.057143
6,Chi-Square,150,0.950000,0.061237
5,Chi-Square,100,0.950000,0.061237
2,Chi-Square,30,0.921429,0.064484
1,Chi-Square,20,0.896429,0.094761


In [38]:
ig_results_train, ig_best_train = evaluate_feature_selection(
    X_train,
    y_encoded,
    method_name="Information Gain",
    score_function=information_gain_score
)

print("\nINFORMATION GAIN EN İYİ SONUÇ")
print(ig_best_train)

Information Gain          k= 10 | Accuracy=0.9179 +/- 0.0674
Information Gain          k= 20 | Accuracy=0.9214 +/- 0.0645
Information Gain          k= 30 | Accuracy=0.9500 +/- 0.0612
Information Gain          k= 50 | Accuracy=0.9500 +/- 0.0612
Information Gain          k= 75 | Accuracy=0.9500 +/- 0.0612
Information Gain          k=100 | Accuracy=0.9500 +/- 0.0612
Information Gain          k=150 | Accuracy=0.9500 +/- 0.0612
Information Gain          k=200 | Accuracy=0.9250 +/- 0.1000
Information Gain          k=300 | Accuracy=0.9500 +/- 0.0612
Information Gain          k=400 | Accuracy=0.9500 +/- 0.0612
Information Gain          k=500 | Accuracy=0.9250 +/- 0.1000

INFORMATION GAIN EN İYİ SONUÇ
Method              Information Gain
k                                 30
CV_Mean_Accuracy                0.95
CV_Std                      0.061237
Name: 2, dtype: object


In [39]:
display(
    ig_results_train.sort_values(
        "CV_Mean_Accuracy",
        ascending=False
    )
)

,Method,k,CV_Mean_Accuracy,CV_Std
2,Information Gain,30,0.950000,0.061237
4,Information Gain,75,0.950000,0.061237
3,Information Gain,50,0.950000,0.061237
8,Information Gain,300,0.950000,0.061237
9,Information Gain,400,0.950000,0.061237
5,Information Gain,100,0.950000,0.061237
6,Information Gain,150,0.950000,0.061237
7,Information Gain,200,0.925000,0.100000
10,Information Gain,500,0.925000,0.100000
1,Information Gain,20,0.921429,0.064484


In [40]:
mi_results_train, mi_best_train = evaluate_feature_selection(
    X_train,
    y_encoded,
    method_name="Mutual Information",
    score_function=mutual_info_classif
)

print("\nMUTUAL INFORMATION EN İYİ SONUÇ")
print(mi_best_train)

Mutual Information        k= 10 | Accuracy=0.8179 +/- 0.0968
Mutual Information        k= 20 | Accuracy=0.9500 +/- 0.1000
Mutual Information        k= 30 | Accuracy=0.9250 +/- 0.0612
Mutual Information        k= 50 | Accuracy=0.9500 +/- 0.0612
Mutual Information        k= 75 | Accuracy=0.9500 +/- 0.0612
Mutual Information        k=100 | Accuracy=0.9500 +/- 0.0612
Mutual Information        k=150 | Accuracy=0.9500 +/- 0.0612
Mutual Information        k=200 | Accuracy=0.9750 +/- 0.0500
Mutual Information        k=300 | Accuracy=0.9750 +/- 0.0500
Mutual Information        k=400 | Accuracy=0.9750 +/- 0.0500
Mutual Information        k=500 | Accuracy=0.9500 +/- 0.1000

MUTUAL INFORMATION EN İYİ SONUÇ
Method              Mutual Information
k                                  200
CV_Mean_Accuracy                 0.975
CV_Std                            0.05
Name: 7, dtype: object


In [41]:
display(
    mi_results_train.sort_values(
        "CV_Mean_Accuracy",
        ascending=False
    )
)

,Method,k,CV_Mean_Accuracy,CV_Std
7,Mutual Information,200,0.975000,0.050000
9,Mutual Information,400,0.975000,0.050000
8,Mutual Information,300,0.975000,0.050000
6,Mutual Information,150,0.950000,0.061237
5,Mutual Information,100,0.950000,0.061237
1,Mutual Information,20,0.950000,0.100000
3,Mutual Information,50,0.950000,0.061237
4,Mutual Information,75,0.950000,0.061237
10,Mutual Information,500,0.950000,0.100000
2,Mutual Information,30,0.925000,0.061237


In [42]:
all_results_train = pd.concat(
    [
        anova_results_train,
        chi2_results_train,
        ig_results_train,
        mi_results_train
    ],
    ignore_index=True
)

display(
    all_results_train.sort_values(
        "CV_Mean_Accuracy",
        ascending=False
    )
)

,Method,k,CV_Mean_Accuracy,CV_Std
20,Chi-Square,400,1.000000,0.000000
19,Chi-Square,300,1.000000,0.000000
6,ANOVA,150,0.975000,0.050000
7,ANOVA,200,0.975000,0.050000
42,Mutual Information,400,0.975000,0.050000
40,Mutual Information,200,0.975000,0.050000
41,Mutual Information,300,0.975000,0.050000
21,Chi-Square,500,0.975000,0.050000
15,Chi-Square,75,0.975000,0.050000
18,Chi-Square,200,0.975000,0.050000


In [43]:
best_results_train = pd.DataFrame([
    anova_best_train,
    chi2_best_train,
    ig_best_train,
    mi_best_train
])

display(
    best_results_train[
        [
            "Method",
            "k",
            "CV_Mean_Accuracy",
            "CV_Std"
        ]
    ].sort_values(
        "CV_Mean_Accuracy",
        ascending=False
    )
)

,Method,k,CV_Mean_Accuracy,CV_Std
8,Chi-Square,300,1.000,0.000000
6,ANOVA,150,0.975,0.050000
7,Mutual Information,200,0.975,0.050000
2,Information Gain,30,0.950,0.061237


In [44]:
def get_selected_features(
    X,
    y,
    score_function,
    best_k
):

    selector = SelectKBest(
        score_func=score_function,
        k=int(best_k)
    )

    # Chi-square ve diğer yöntemler için
    # aynı ölçekleme mantığını kullan
    scaler = MinMaxScaler()

    X_scaled = scaler.fit_transform(X)

    X_selected = selector.fit_transform(
        X_scaled,
        y
    )

    selected_features = X.columns[
        selector.get_support()
    ]

    return (
        X_selected,
        selected_features,
        selector,
        scaler
    )

In [45]:
best_k_anova = int(anova_best_train["k"])
best_k_chi2 = int(chi2_best_train["k"])
best_k_ig = int(ig_best_train["k"])
best_k_mi = int(mi_best_train["k"])

X_anova, genes_anova, _, _ = get_selected_features(
    X_train,
    y_encoded,
    f_classif,
    best_k_anova
)

X_chi2, genes_chi2, _, _ = get_selected_features(
    X_train,
    y_encoded,
    chi2,
    best_k_chi2
)

X_ig, genes_ig, _, _ = get_selected_features(
    X_train,
    y_encoded,
    information_gain_score,
    best_k_ig
)

X_mi, genes_mi, _, _ = get_selected_features(
    X_train,
    y_encoded,
    mutual_info_classif,
    best_k_mi
)

print("ANOVA gen sayısı:", len(genes_anova))
print("Chi-Square gen sayısı:", len(genes_chi2))
print("Information Gain gen sayısı:", len(genes_ig))
print("Mutual Information gen sayısı:", len(genes_mi))

ANOVA gen sayısı: 150
Chi-Square gen sayısı: 300
Information Gain gen sayısı: 30
Mutual Information gen sayısı: 200


In [46]:
all_results_train.to_csv(
    "feature_selection_cv_results_train.csv",
    index=False
)

best_results_train.to_csv(
    "best_k_results_train.csv",
    index=False
)

pd.DataFrame({
    "ANOVA": pd.Series(genes_anova),
    "Chi-Square": pd.Series(genes_chi2),
    "Information Gain": pd.Series(genes_ig),
    "Mutual Information": pd.Series(genes_mi)
}).to_csv(
    "selected_genes_train.csv",
    index=False
)

print("Sonuç dosyaları kaydedildi.")

Sonuç dosyaları kaydedildi.


In [47]:
arm2_path = "/kaggle/input/datasets/lknurkoparr/armstrong-dataset/Arm2.csv"

arm2_df = pd.read_csv(arm2_path)

print("Arm2 boyutu:", arm2_df.shape)

display(arm2_df.head())

Arm2 boyutu: (72, 2196)


,Unnamed: 0,AFFX-MurIL10_at,AFFX-PheX-3_at,AFFX-HUMISGF3A/M97935_5_at,AFFX-HUMISGF3A/M97935_MA_at,AFFX-HUMRGE/M10098_5_at,AFFX-HUMRGE/M10098_M_at,AFFX-HUMTFRR/M11507_M_at,AFFX-M27830_5_at,31312_at,...,115_at,118_at,120_at,122_at,123_at,135_g_at,101_at,103_at,107_at,group
0,ALL,-0.178903,0.699058,1.570324,1.169136,0.454937,0.974153,0.791923,1.953105,-0.994379,...,-0.901997,-0.678152,0.539615,1.223385,0.270060,-0.886159,-0.817232,0.289284,0.343642,1
1,ALL.1,-1.209637,1.129464,0.703637,0.873118,-0.747602,-0.305183,0.176481,0.287177,1.612543,...,-0.901997,1.177254,0.080988,1.088827,-1.237540,-0.886159,-0.817232,0.247350,0.225096,1
2,ALL.2,0.615612,-0.955412,-1.085843,-0.939503,1.293281,2.262446,0.820937,2.391259,-0.994379,...,-0.901997,-0.678152,-0.126719,-0.916671,0.144569,-0.886159,-0.817232,-1.397007,0.091560,1
3,ALL.3,0.809085,1.153651,-0.489633,-1.799324,-0.747602,-0.305183,-1.023976,-0.837095,-0.046946,...,0.104273,-0.678152,-1.217400,-0.916671,-1.237540,0.511934,1.817364,-1.397007,0.301060,1
4,ALL.4,0.897607,1.199732,1.679592,1.490304,-0.747602,-0.305183,0.393708,0.050672,1.256111,...,-0.156733,-0.678152,0.501654,1.315438,0.182204,-0.886159,0.963845,0.595650,1.319559,1


In [53]:
# Sınıf etiketi
y_arm2 = arm2_df["group"].copy()

# Hasta ID ve sınıf sütunlarını özelliklerden çıkar
drop_columns = [
    "Unnamed: 0",
    "group"
]

X_arm2 = arm2_df.drop(
    columns=drop_columns,
    errors="ignore"
)

# Bütün genleri sayısal yap
X_arm2 = X_arm2.apply(
    pd.to_numeric,
    errors="coerce"
)

# Eksik değerleri medyan ile doldur
X_arm2 = X_arm2.fillna(
    X_arm2.median()
)

print("X_arm2 boyutu:", X_arm2.shape)

print("\ny_arm2 sınıf dağılımı:")
print(y_arm2.value_counts().sort_index())

X_arm2 boyutu: (72, 2194)

y_arm2 sınıf dağılımı:
group
1    24
2    20
3    28
Name: count, dtype: int64


In [54]:
print("Orijinal sınıflar:")
print(sorted(y_arm2.unique()))

print("\nSınıf sayıları:")
print(y_arm2.value_counts().sort_index())

Orijinal sınıflar:
[np.int64(1), np.int64(2), np.int64(3)]

Sınıf sayıları:
group
1    24
2    20
3    28
Name: count, dtype: int64


In [56]:
def information_gain_score(X, y):
    """
    Çok sınıflı sınıflandırma için Information Gain.
    Sürekli gen ekspresyon değerleri quantile binning
    ile kategorik hale getirilir.
    """

    X = np.asarray(X)
    y = np.asarray(y)

    n_features = X.shape[1]
    scores = np.zeros(n_features)

    for j in range(n_features):

        feature = X[:, j]

        # Eksik değer kontrolü
        if np.isnan(feature).any():
            feature = np.nan_to_num(
                feature,
                nan=np.nanmedian(feature)
            )

        # Sabit gen
        if np.ptp(feature) == 0:
            scores[j] = 0
            continue

        try:
            # 5 quantile bin
            bins = np.quantile(
                feature,
                [0, 0.2, 0.4, 0.6, 0.8, 1.0]
            )

            bins = np.unique(bins)

            if len(bins) < 3:
                scores[j] = 0
                continue

            feature_discrete = np.digitize(
                feature,
                bins[1:-1],
                right=False
            )

            # Information Gain / Mutual Information
            scores[j] = mutual_info_score(
                y,
                feature_discrete
            )

        except Exception:
            scores[j] = 0

    return scores

In [57]:
def evaluate_feature_selection(
    X,
    y,
    method_name,
    score_function,
    k_values=K_VALUES,
    cv=CV
):

    results = []

    for k in k_values:

        if k > X.shape[1]:
            continue

        pipeline = Pipeline([
            (
                "scaler",
                MinMaxScaler()
            ),

            (
                "feature_selection",
                SelectKBest(
                    score_func=score_function,
                    k=k
                )
            ),

            (
                "classifier",
                LinearSVC(
                    C=1.0,
                    random_state=RANDOM_STATE,
                    max_iter=20000
                )
            )
        ])

        scores = cross_val_score(
            pipeline,
            X,
            y,
            cv=cv,
            scoring="accuracy",
            n_jobs=-1
        )

        results.append({
            "Method": method_name,
            "k": k,
            "CV_Mean_Accuracy": scores.mean(),
            "CV_Std": scores.std()
        })

        print(
            f"{method_name:25s} "
            f"k={k:3d} | "
            f"Accuracy={scores.mean():.4f} "
            f"+/- {scores.std():.4f}"
        )

    results_df = pd.DataFrame(results)

    best_row = results_df.loc[
        results_df["CV_Mean_Accuracy"].idxmax()
    ]

    return results_df, best_row

In [58]:
anova_results_arm2, anova_best_arm2 = evaluate_feature_selection(
    X_arm2,
    y_arm2,
    method_name="ANOVA",
    score_function=f_classif
)

print("\n================================")
print("ARM2 - ANOVA EN İYİ SONUÇ")
print("================================")

print(anova_best_arm2)

ANOVA                     k= 10 | Accuracy=0.9581 +/- 0.0343
ANOVA                     k= 20 | Accuracy=0.9581 +/- 0.0343
ANOVA                     k= 30 | Accuracy=0.9714 +/- 0.0350
ANOVA                     k= 50 | Accuracy=0.9714 +/- 0.0350
ANOVA                     k= 75 | Accuracy=0.9581 +/- 0.0343
ANOVA                     k=100 | Accuracy=0.9714 +/- 0.0350
ANOVA                     k=150 | Accuracy=0.9714 +/- 0.0350
ANOVA                     k=200 | Accuracy=0.9714 +/- 0.0350
ANOVA                     k=300 | Accuracy=0.9714 +/- 0.0350
ANOVA                     k=400 | Accuracy=0.9714 +/- 0.0350
ANOVA                     k=500 | Accuracy=0.9714 +/- 0.0350

ARM2 - ANOVA EN İYİ SONUÇ
Method                 ANOVA
k                         30
CV_Mean_Accuracy    0.971429
CV_Std              0.034993
Name: 2, dtype: object


In [59]:
chi2_results_arm2, chi2_best_arm2 = evaluate_feature_selection(
    X_arm2,
    y_arm2,
    method_name="Chi-Square",
    score_function=chi2
)

print("\n================================")
print("ARM2 - CHI-SQUARE EN İYİ SONUÇ")
print("================================")

print(chi2_best_arm2)

Chi-Square                k= 10 | Accuracy=0.8886 +/- 0.0570
Chi-Square                k= 20 | Accuracy=0.9314 +/- 0.0422
Chi-Square                k= 30 | Accuracy=0.9581 +/- 0.0343
Chi-Square                k= 50 | Accuracy=0.9581 +/- 0.0343
Chi-Square                k= 75 | Accuracy=0.9581 +/- 0.0343
Chi-Square                k=100 | Accuracy=0.9724 +/- 0.0339
Chi-Square                k=150 | Accuracy=0.9724 +/- 0.0339
Chi-Square                k=200 | Accuracy=0.9724 +/- 0.0339
Chi-Square                k=300 | Accuracy=0.9857 +/- 0.0286
Chi-Square                k=400 | Accuracy=0.9857 +/- 0.0286
Chi-Square                k=500 | Accuracy=0.9857 +/- 0.0286

ARM2 - CHI-SQUARE EN İYİ SONUÇ
Method              Chi-Square
k                          300
CV_Mean_Accuracy      0.985714
CV_Std                0.028571
Name: 8, dtype: object


In [60]:
ig_results_arm2, ig_best_arm2 = evaluate_feature_selection(
    X_arm2,
    y_arm2,
    method_name="Information Gain",
    score_function=information_gain_score
)

print("\n================================")
print("ARM2 - INFORMATION GAIN EN İYİ SONUÇ")
print("================================")

print(ig_best_arm2)

Information Gain          k= 10 | Accuracy=0.9448 +/- 0.0504
Information Gain          k= 20 | Accuracy=0.9438 +/- 0.0532
Information Gain          k= 30 | Accuracy=0.9714 +/- 0.0350
Information Gain          k= 50 | Accuracy=0.9714 +/- 0.0350
Information Gain          k= 75 | Accuracy=0.9714 +/- 0.0350
Information Gain          k=100 | Accuracy=0.9714 +/- 0.0350
Information Gain          k=150 | Accuracy=0.9714 +/- 0.0350
Information Gain          k=200 | Accuracy=0.9714 +/- 0.0350
Information Gain          k=300 | Accuracy=0.9714 +/- 0.0350
Information Gain          k=400 | Accuracy=0.9714 +/- 0.0350
Information Gain          k=500 | Accuracy=0.9714 +/- 0.0350

ARM2 - INFORMATION GAIN EN İYİ SONUÇ
Method              Information Gain
k                                 30
CV_Mean_Accuracy            0.971429
CV_Std                      0.034993
Name: 2, dtype: object


In [61]:
mi_results_arm2, mi_best_arm2 = evaluate_feature_selection(
    X_arm2,
    y_arm2,
    method_name="Mutual Information",
    score_function=mutual_info_classif
)

print("\n================================")
print("ARM2 - MUTUAL INFORMATION EN İYİ SONUÇ")
print("================================")

print(mi_best_arm2)

Mutual Information        k= 10 | Accuracy=0.9010 +/- 0.1327
Mutual Information        k= 20 | Accuracy=0.9448 +/- 0.0504
Mutual Information        k= 30 | Accuracy=0.9714 +/- 0.0350
Mutual Information        k= 50 | Accuracy=0.9714 +/- 0.0350
Mutual Information        k= 75 | Accuracy=0.9714 +/- 0.0350
Mutual Information        k=100 | Accuracy=0.9714 +/- 0.0350
Mutual Information        k=150 | Accuracy=0.9714 +/- 0.0350
Mutual Information        k=200 | Accuracy=0.9714 +/- 0.0350
Mutual Information        k=300 | Accuracy=0.9714 +/- 0.0350
Mutual Information        k=400 | Accuracy=0.9714 +/- 0.0350
Mutual Information        k=500 | Accuracy=0.9714 +/- 0.0350

ARM2 - MUTUAL INFORMATION EN İYİ SONUÇ
Method              Mutual Information
k                                   30
CV_Mean_Accuracy              0.971429
CV_Std                        0.034993
Name: 2, dtype: object


In [62]:
all_results_arm2 = pd.concat(
    [
        anova_results_arm2,
        chi2_results_arm2,
        ig_results_arm2,
        mi_results_arm2
    ],
    ignore_index=True
)

display(
    all_results_arm2.sort_values(
        "CV_Mean_Accuracy",
        ascending=False
    )
)

,Method,k,CV_Mean_Accuracy,CV_Std
20,Chi-Square,400,0.985714,0.028571
21,Chi-Square,500,0.985714,0.028571
19,Chi-Square,300,0.985714,0.028571
18,Chi-Square,200,0.972381,0.033860
17,Chi-Square,150,0.972381,0.033860
16,Chi-Square,100,0.972381,0.033860
10,ANOVA,500,0.971429,0.034993
9,ANOVA,400,0.971429,0.034993
3,ANOVA,50,0.971429,0.034993
2,ANOVA,30,0.971429,0.034993


In [63]:
best_results_arm2 = pd.DataFrame([
    anova_best_arm2,
    chi2_best_arm2,
    ig_best_arm2,
    mi_best_arm2
])

best_results_arm2 = best_results_arm2[
    [
        "Method",
        "k",
        "CV_Mean_Accuracy",
        "CV_Std"
    ]
].sort_values(
    "CV_Mean_Accuracy",
    ascending=False
)

display(best_results_arm2)

,Method,k,CV_Mean_Accuracy,CV_Std
8,Chi-Square,300,0.985714,0.028571
2,ANOVA,30,0.971429,0.034993
2,Information Gain,30,0.971429,0.034993
2,Mutual Information,30,0.971429,0.034993


## **HİBRİT TABANLI ÖZNİTELİK SEÇİM YÖNTEMLERİ**

In [3]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_selection import SelectKBest, RFE
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report

In [4]:
RANDOM_STATE = 42
CV_SPLITS = 5

# Information Gain aşamasında tutulacak ilk aday gen sayısı.
# GridSearchCV bunun ardından SVM-RFE ile daha küçük alt kümeler dener.
IG_TOP_N = 200

# GridSearchCV'nin deneyeceği nihai özellik sayıları.
FEATURE_COUNTS = [5, 10, 15, 20, 30, 40, 50, 75, 100]

In [7]:
# ------------------------------------------------------------
# 2. INFORMATION GAIN SKOR FONKSİYONU
# ------------------------------------------------------------

def information_gain_score(X, y):
    """
    Sürekli gen ifade değerlerini quantile binning ile ayrıklaştırır
    ve her gen için sınıf etiketiyle mutual information hesaplar.
    Bu skor, filtre aşamasında Information Gain olarak kullanılır.
    """
    from sklearn.metrics import mutual_info_score

    X = np.asarray(X)
    y = np.asarray(y)

    scores = np.zeros(X.shape[1], dtype=float)

    for j in range(X.shape[1]):
        feature = X[:, j]

        if np.ptp(feature) == 0:
            scores[j] = 0.0
            continue

        try:
            bins = np.quantile(
                feature,
                [0, 0.2, 0.4, 0.6, 0.8, 1.0]
            )
            bins = np.unique(bins)

            if len(bins) < 3:
                scores[j] = 0.0
                continue

            feature_discrete = np.digitize(
                feature,
                bins[1:-1],
                right=False
            )

            scores[j] = mutual_info_score(y, feature_discrete)

        except Exception:
            scores[j] = 0.0

    return scores

In [8]:
# ------------------------------------------------------------
# 3. VERİ SETLERİNİ YÜKLEME
# ------------------------------------------------------------

# ALL_AML dosya yolları
ALL_AML_TRAIN_PATH = (
    "/kaggle/input/datasets/crawford/gene-expression/"
    "data_set_ALL_AML_train.csv"
)
ALL_AML_LABEL_PATH = (
    "/kaggle/input/datasets/crawford/gene-expression/actual.csv"
)

# Arm2 dosya yolu
ARM2_PATH = (
    "/kaggle/input/datasets/lknurkoparr/armstrong-dataset/Arm2.csv"
)


def load_all_aml(train_path, label_path):
    train_df = pd.read_csv(train_path)
    actual_df = pd.read_csv(label_path)

    # Hasta numarası olan sütunları belirle
    patient_columns = [
        col for col in train_df.columns if str(col).isdigit()
    ]

    # Satırlar gen, sütunlar hasta olacak şekilde transpoze et
    X = train_df[patient_columns].apply(
        pd.to_numeric, errors="coerce"
    ).T

    X.index = X.index.astype(int)

    # Etiketleri hasta numaralarıyla eşleştir
    actual_df["patient"] = pd.to_numeric(
        actual_df["patient"], errors="coerce"
    )

    y_df = actual_df[
        actual_df["patient"].isin(X.index)
    ].copy()

    y_df = y_df.set_index("patient")
    y_df = y_df.loc[X.index]

    y = y_df["cancer"].map({"ALL": 0, "AML": 1})

    valid = y.notna()
    X = X.loc[valid]
    y = y.loc[valid].astype(int)

    # Tamamen boş gen sütunlarını kaldır
    X = X.dropna(axis=1, how="all")

    return X, y


def load_arm2(path):
    df = pd.read_csv(path)

    y = df["group"].copy()

    X = df.drop(
        columns=["Unnamed: 0", "group"],
        errors="ignore"
    )

    X = X.apply(pd.to_numeric, errors="coerce")

    # Tamamen boş gen sütunlarını kaldır
    X = X.dropna(axis=1, how="all")

    return X, y

In [11]:
# ------------------------------------------------------------
# 4. GRID SEARCH: INFORMATION GAIN + SVM-RFE
# ------------------------------------------------------------

def run_hybrid_grid_search(X, y, dataset_name):
    """
    Her CV eğitim katında:
      1) Eksik değerleri doldurur.
      2) Ölçekleme yapar.
      3) Information Gain ile IG_TOP_N gen seçer.
      4) SVM-RFE ile GridSearchCV'nin belirlediği sayıda gen seçer.
      5) Linear SVM sınıflandırıcısını eğitir.

    Özellik seçimi Pipeline içinde olduğundan CV doğrulama verisi
    özellik seçimi aşamasında kullanılmaz.
    """

    X = X.copy()
    y = pd.Series(y).reset_index(drop=True)
    X = X.reset_index(drop=True)

    # Sabit veya tek değerli sütunları kaldır
    nunique = X.nunique(dropna=True)
    X = X.loc[:, nunique > 1]

    # Information Gain aday sayısını mevcut özellik sayısına uyarla
    effective_ig_n = min(IG_TOP_N, X.shape[1])

    # Grid'de yalnızca IG aday havuzundan küçük/eşit değerler kullan
    feature_grid = [
        k for k in FEATURE_COUNTS if k <= effective_ig_n
    ]

    if not feature_grid:
        raise ValueError(
            f"{dataset_name}: Grid için uygun özellik sayısı yok. "
            f"Mevcut IG aday sayısı: {effective_ig_n}"
        )

    # Sınıf başına örnek sayısı CV kat sayısından azsa hata oluşmasını önle
    min_class_count = int(y.value_counts().min())
    n_splits = min(CV_SPLITS, min_class_count)

    if n_splits < 2:
        raise ValueError(
            f"{dataset_name}: Stratified CV için her sınıfta "
            "en az 2 örnek bulunmalıdır."
        )

    cv = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=RANDOM_STATE
    )

    pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),

        # Filtre yöntemi: Information Gain
        ("filter", SelectKBest(
            score_func=information_gain_score,
            k=effective_ig_n
        )),

        # Gömülü/Wrapper tabanlı aşama: SVM-RFE
        ("rfe", RFE(
            estimator=LinearSVC(
                C=1.0,
                dual="auto",
                max_iter=20000,
                random_state=RANDOM_STATE
            ),
            step=0.1
        )),

        # Son sınıflandırıcı
        ("classifier", LinearSVC(
            C=1.0,
            dual="auto",
            max_iter=20000,
            random_state=RANDOM_STATE
        ))
    ])

    param_grid = {
        "rfe__n_features_to_select": feature_grid
    }

    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring="accuracy",
        cv=cv,
        n_jobs=-1,
        refit=True,
        return_train_score=True
    )

    grid_search.fit(X, y)

    # Grid Search sonuç tablosu
    results = pd.DataFrame(grid_search.cv_results_)

    results_df = pd.DataFrame({
        "Dataset": dataset_name,
        "IG_Top_N": effective_ig_n,
        "Selected_Features": results["param_rfe__n_features_to_select"].astype(int),
        "Mean_CV_Accuracy": results["mean_test_score"],
        "Std_CV_Accuracy": results["std_test_score"],
        "Mean_Train_Accuracy": results["mean_train_score"],
        "Rank": results["rank_test_score"]
    }).sort_values(
        ["Mean_CV_Accuracy", "Std_CV_Accuracy"],
        ascending=[False, True]
    ).reset_index(drop=True)

    print("\n" + "=" * 65)
    print(f"{dataset_name} - INFORMATION GAIN + SVM-RFE")
    print("=" * 65)
    print("Özellik sayısı için Grid:", feature_grid)
    print("En iyi parametre:", grid_search.best_params_)
    print(f"En iyi CV Accuracy: {grid_search.best_score_:.4f}")

    display(results_df)

    # En iyi pipeline tüm veri üzerinde yeniden eğitilmiştir (refit=True).
    # Bu modelin seçtiği gen isimlerini çıkar.
    best_pipeline = grid_search.best_estimator_

    filter_support = best_pipeline.named_steps["filter"].get_support()
    rfe_support = best_pipeline.named_steps["rfe"].get_support()

    ig_feature_names = X.columns[filter_support]
    selected_feature_names = ig_feature_names[rfe_support]

    selected_features_df = pd.DataFrame({
        "Selected_Gene": selected_feature_names
    })

    print("\nSeçilen gen sayısı:", len(selected_feature_names))
    print("Seçilen genler:")
    display(selected_features_df)

    return grid_search, results_df, selected_features_df


In [13]:
# ------------------------------------------------------------
# 5. ALL_AML VERİ SETİ
# ------------------------------------------------------------

X_all_aml, y_all_aml = load_all_aml(
    ALL_AML_TRAIN_PATH,
    ALL_AML_LABEL_PATH
)

print("ALL_AML X boyutu:", X_all_aml.shape)
print("ALL_AML sınıf dağılımı:")
print(y_all_aml.value_counts())

all_aml_grid, all_aml_results, all_aml_genes = (
    run_hybrid_grid_search(
        X_all_aml,
        y_all_aml,
        dataset_name="ALL_AML"
    )
)

ALL_AML X boyutu: (38, 7129)
ALL_AML sınıf dağılımı:
cancer
0    27
1    11
Name: count, dtype: int64

ALL_AML - INFORMATION GAIN + SVM-RFE
Özellik sayısı için Grid: [5, 10, 15, 20, 30, 40, 50, 75, 100]
En iyi parametre: {'rfe__n_features_to_select': 30}
En iyi CV Accuracy: 0.9500


,Dataset,IG_Top_N,Selected_Features,Mean_CV_Accuracy,Std_CV_Accuracy,Mean_Train_Accuracy,Rank
0,ALL_AML,200,30,0.950000,0.061237,1.0,1
1,ALL_AML,200,40,0.950000,0.061237,1.0,1
2,ALL_AML,200,50,0.950000,0.061237,1.0,1
3,ALL_AML,200,75,0.950000,0.061237,1.0,1
4,ALL_AML,200,100,0.950000,0.061237,1.0,1
5,ALL_AML,200,20,0.921429,0.064484,1.0,6
6,ALL_AML,200,15,0.896429,0.094761,1.0,7
7,ALL_AML,200,10,0.892857,0.054163,1.0,8
8,ALL_AML,200,5,0.842857,0.121323,1.0,9



Seçilen gen sayısı: 30
Seçilen genler:


,Selected_Gene
0,311
1,320
2,1132
3,1206
4,1744
5,1778
6,1828
7,1833
8,1881
9,1923


In [14]:
# ------------------------------------------------------------
# 6. ARM2 VERİ SETİ
# ------------------------------------------------------------

X_arm2, y_arm2 = load_arm2(ARM2_PATH)

print("\nArm2 X boyutu:", X_arm2.shape)
print("Arm2 sınıf dağılımı:")
print(y_arm2.value_counts())

arm2_grid, arm2_results, arm2_genes = (
    run_hybrid_grid_search(
        X_arm2,
        y_arm2,
        dataset_name="Arm2"
    )
)


Arm2 X boyutu: (72, 2194)
Arm2 sınıf dağılımı:
group
3    28
1    24
2    20
Name: count, dtype: int64

Arm2 - INFORMATION GAIN + SVM-RFE
Özellik sayısı için Grid: [5, 10, 15, 20, 30, 40, 50, 75, 100]
En iyi parametre: {'rfe__n_features_to_select': 20}
En iyi CV Accuracy: 0.9714


,Dataset,IG_Top_N,Selected_Features,Mean_CV_Accuracy,Std_CV_Accuracy,Mean_Train_Accuracy,Rank
0,Arm2,200,20,0.971429,0.034993,1.000000,1
1,Arm2,200,30,0.971429,0.034993,1.000000,1
2,Arm2,200,40,0.971429,0.034993,1.000000,1
3,Arm2,200,50,0.971429,0.034993,1.000000,1
4,Arm2,200,75,0.971429,0.034993,1.000000,1
5,Arm2,200,100,0.971429,0.034993,1.000000,1
6,Arm2,200,10,0.958095,0.034259,1.000000,7
7,Arm2,200,15,0.957143,0.057143,1.000000,8
8,Arm2,200,5,0.875238,0.069361,0.972232,9



Seçilen gen sayısı: 20
Seçilen genler:


,Selected_Gene
0,41465_at
1,33320_at
2,35164_at
3,36873_at
4,37187_at
5,37280_at
6,37908_at
7,39385_at
8,40763_at
9,40782_at


In [15]:
# ------------------------------------------------------------
# 7. SONUÇLARI BİRLEŞTİRME VE KAYDETME
# ------------------------------------------------------------

all_results = pd.concat(
    [all_aml_results, arm2_results],
    ignore_index=True
)

best_results = pd.DataFrame([
    {
        "Dataset": "ALL_AML",
        "Best_Feature_Count": all_aml_grid.best_params_[
            "rfe__n_features_to_select"
        ],
        "Best_CV_Accuracy": all_aml_grid.best_score_
    },
    {
        "Dataset": "Arm2",
        "Best_Feature_Count": arm2_grid.best_params_[
            "rfe__n_features_to_select"
        ],
        "Best_CV_Accuracy": arm2_grid.best_score_
    }
])

display(all_results)
display(best_results)

all_results.to_csv(
    "IG_SVMRFE_gridsearch_all_results.csv",
    index=False
)

best_results.to_csv(
    "IG_SVMRFE_gridsearch_best_results.csv",
    index=False
)

all_aml_genes.to_csv(
    "IG_SVMRFE_ALL_AML_selected_genes.csv",
    index=False
)

arm2_genes.to_csv(
    "IG_SVMRFE_Arm2_selected_genes.csv",
    index=False
)

print("\nTüm sonuçlar CSV dosyalarına kaydedildi.")

,Dataset,IG_Top_N,Selected_Features,Mean_CV_Accuracy,Std_CV_Accuracy,Mean_Train_Accuracy,Rank
0,ALL_AML,200,30,0.950000,0.061237,1.000000,1
1,ALL_AML,200,40,0.950000,0.061237,1.000000,1
2,ALL_AML,200,50,0.950000,0.061237,1.000000,1
3,ALL_AML,200,75,0.950000,0.061237,1.000000,1
4,ALL_AML,200,100,0.950000,0.061237,1.000000,1
5,ALL_AML,200,20,0.921429,0.064484,1.000000,6
6,ALL_AML,200,15,0.896429,0.094761,1.000000,7
7,ALL_AML,200,10,0.892857,0.054163,1.000000,8
8,ALL_AML,200,5,0.842857,0.121323,1.000000,9
9,Arm2,200,20,0.971429,0.034993,1.000000,1


,Dataset,Best_Feature_Count,Best_CV_Accuracy
0,ALL_AML,30,0.950000
1,Arm2,20,0.971429



Tüm sonuçlar CSV dosyalarına kaydedildi.
